# AI CUP 2026 圍棋（競賽一＋競賽二）：第 1 層重現（Colab，只用 CPU，公開 repo 版）

從**已存的中間特徵**（CNN 向量、KataGo P1 特徵、各層模型輸出）重算兩場最終交件 CSV，和已上傳的檔比 SHA-256。

| 場次 | 入口 | 預期 SHA-256 |
|---|---|---|
| 競賽二（預測棋力，交件 69） | `repro/comp2_from_features.py` | `c7595f2eb43c185973c55c7b4b03875de41c5ce92327f7042fc3769dbbae36c8` |
| 競賽一（辨識玩家，交件 v2-9） | `repro/comp1_from_features.py` | `36b222d866267abffda03d126d94db9ed5bea3406b2f8b47323622fd5329a026` |

**來源有三個**：

1. 程式：公開 repo（下一格的 `REPO_URL`），用 `git clone` 取得。
2. **官方測試檔（請自行從 AIdea 下載；官方資料不得轉載，本專案的任何包裡都沒有）**：到兩場競賽頁面的「資料」分頁下載下面兩個 zip，放進自己 Google Drive 的一個資料夾（下一格的 `OFFICIAL_DIR`；放 zip 或解開的 CSV 都可以，會依檔名尋找並核對 SHA-256）：
   - 競賽一 v2 測試集 `aicup2026_player_identification_v2_tests_20261108_fix_readme.zip`（MD5 `90cda19fc750f0c99ab75c0706ba9314`）：`player_identification_{candidates,candidates2,test,test2}_{public,private}.csv` 共 8 個
   - 競賽二 `aicup2026_rank_prediction_tests_20261030.zip`（MD5 `355c0b40be6af0413859991ebbcf31f3`）：`rank_prediction_test_{public,private}.csv` 共 2 個
   - 10 個 CSV 各自的 bytes、SHA-256、MD5 列在 repo 的 `repro/inputs_public.json`（`"from": "official"`）與 README 第 4 節。
3. 特徵：雲端硬碟上的三個包（都不含官方檔）。把雲端資料夾加到自己的 Google Drive（或下載後上傳），在下一格把 `DRIVE_DIR` 指到它：
   - `C10_public_features_comp1.zip`（897 個檔，4,598,622,322 bytes）
   - `C10_public_features_comp2.zip`（56 個檔，120,212,732 bytes）
   - `C10_features_layer1_ingit.zip`（67 個檔，71,507,868 bytes）
   - 資料夾裡放的若是 7 MB 分塊（`parts/` ＋ `C10_public_parts_manifest.json`），notebook 會先接回並逐塊、整檔核 SHA-256。

執行階段選一般 CPU 即可（不需要 GPU；建議「高 RAM」；本機磁碟約需 11 GB）。由上到下「全部執行」。

流程：clone repo → 取得三個包 → `repro/stage_inputs.py`（核官方測試檔的 SHA-256、複製到 repo 原路徑，並由它們重建 6 個衍生檔（改名副本、只有表頭的候選檔、拿掉開頭 HD 的題目檔）→ 核每個包與每個檔的 SHA-256 → 解壓到 repo 原路徑 → 把 `C10_features_layer1_ingit.zip` 的 67 個檔在**這個 clone 裡** `git add -f` 並 commit）→ 用 `uv` 建 Python 3.12.13 環境 → 兩支入口各先跑嚴格（逐位元組）模式，停在接縫就加 `--tolerant` 再跑 → 比 SHA-256。

說明：
- 這裡不做任何訓練或特徵抽取，不呼叫 GPU；方法都在 repo 既有的程式裡，入口只負責呼叫與比對。
- 為什麼要在 clone 裡 commit：競賽二的凍結工具要求輸入檔在 git 裡已追蹤而且和 HEAD 相同（`tools/t25_rank.py` 的 `require_committed`）。這個 commit 只存在於 Colab 的 clone，不會也不應該被推回去。
- 凍結工具會比對 Python 3.12.13、numpy 2.5.3、scipy 1.18.1，所以用 `uv` 另建環境，不用 Colab 內建的 Python。
- 競賽二的工具要求**不要設** `OMP_NUM_THREADS`／`OPENBLAS_NUM_THREADS`／`MKL_NUM_THREADS`；啟動入口時會把這三個變數從子程序的環境拿掉。競賽一的入口在 Linux 上會在程序內把 numpy 的 OpenBLAS 限成 1 執行緒（原本在 Windows 上既有程式就是這樣跑的）。
- 原本的結果是在 Windows 上算的；Linux 的浮點數最後一位可能不同，凍結工具的逐位元組接縫會先停下、不產生最終 CSV。這時改用 `--tolerant`：只在入口程序內把精確比較換成容許差比較（`repro/_tolerant.py`；數值接縫 |差| ≤ 1e-9 + 1e-9·|參考值|，接縫自己有容許差就用它的；超過仍停），每個接縫的實際最大差異寫進 `tolerant_report.json`。
- **PASS 的判定不變**：只看最終 CSV 的 SHA-256 是否等於已上傳檔。不同時會印出和已上傳檔有幾列不同與前 10 列。
- 交付包不含驗證集答案（`data/local`）；`stage_inputs.py` 遇到這個路徑就停。

In [ ]:
# 0. 參數
REPO_URL = "https://github.com/kuotunyu/aicup2026-go.git"   # 公開 repo
REPO_REF = ""                  # 要固定某個 commit／tag 就填；空白＝預設分支的最新 commit
MOUNT_DRIVE = True             # False：不掛 Drive，DRIVE_DIR 改填 Colab 本機上已放好三個 zip 的資料夾
DRIVE_DIR = "/content/drive/MyDrive/C10/public_repro"   # 放三個特徵包（或 parts/ 分塊＋C10_public_parts_manifest.json）的資料夾
OFFICIAL_DIR = "/content/drive/MyDrive/C10/official_tests"   # 你自己從 AIdea 下載的官方測試檔（兩個 zip 或 10 個 CSV）
SAVE_TO_DRIVE = True           # 跑完把紀錄和產生的 CSV 存到 DRIVE_DIR/colab_run_<時間>/（資料夾唯讀時自動略過）
WORK = "/content/c10"          # clone 出來的 repo 樹
OUT = "/content/out"           # 輸出（必須在 repo 樹之外）
VENV = "/content/venv312"      # Python 3.12.13 環境（在 repo 樹之外）
ZIP_LOCAL = "/content/zips"    # 三個 zip 複製／接回到本機磁碟的位置
ZIPS = ["C10_public_features_comp1.zip", "C10_public_features_comp2.zip", "C10_features_layer1_ingit.zip"]
PARTS_MANIFEST = "C10_public_parts_manifest.json"   # 分塊上傳時的清單（和 parts/ 一起放在 DRIVE_DIR）

import time
T_START = time.time()
TIMES = {}

In [ ]:
# 1. clone 公開 repo（保留 git 的原始位元組：core.autocrlf=false）
import os, shutil, subprocess, json, hashlib
t0 = time.time()
assert not os.path.exists(WORK), f"{WORK} 已存在；要重跑請先「中斷連線並刪除執行階段」，或手動刪掉它"
r = subprocess.run(["git", "-c", "core.autocrlf=false", "clone", "-q", REPO_URL, WORK], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
subprocess.run(["git", "-C", WORK, "config", "core.autocrlf", "false"], check=True)
if REPO_REF:
    r = subprocess.run(["git", "-C", WORK, "checkout", "-q", REPO_REF], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr
HEAD = subprocess.run(["git", "-C", WORK, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert os.path.isfile(os.path.join(WORK, "repro", "stage_inputs.py")), "這個 commit 沒有 repro/stage_inputs.py"
with open(os.path.join(WORK, "repro", "inputs_public.json"), encoding="utf-8") as f:
    MAN = json.load(f)
assert sorted(MAN["zips"]) == sorted(ZIPS), f"repo 的 inputs_public.json 列的 zip 是 {sorted(MAN['zips'])}"
TIMES["clone"] = time.time() - t0
print(f"{REPO_URL} @ {HEAD} -> {WORK}（{TIMES['clone']:.0f} 秒）")
for n in ZIPS:
    print(f"  需要 {n}: {MAN['zips'][n]['bytes']:,} bytes，SHA-256 {MAN['zips'][n]['sha256']}")

In [ ]:
# 2. 取得三個特徵包：DRIVE_DIR 有整檔就複製到本機磁碟；沒有就用 parts/ 的 7 MB 分塊接回（逐塊、整檔核 SHA-256）
t0 = time.time()
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
assert os.path.isdir(DRIVE_DIR), f"找不到資料夾 {DRIVE_DIR}（改第 0 格的 DRIVE_DIR）"
assert os.path.isdir(OFFICIAL_DIR), f"找不到資料夾 {OFFICIAL_DIR}：請先從 AIdea 下載官方測試檔放進去（改第 0 格的 OFFICIAL_DIR）"
os.makedirs(ZIP_LOCAL, exist_ok=True)
pm = {}
if os.path.isfile(os.path.join(DRIVE_DIR, PARTS_MANIFEST)):
    with open(os.path.join(DRIVE_DIR, PARTS_MANIFEST), encoding="utf-8") as f:
        pm = json.load(f)
for name in ZIPS:
    want = MAN["zips"][name]
    whole, local = os.path.join(DRIVE_DIR, name), os.path.join(ZIP_LOCAL, name)
    if os.path.isfile(local) and os.path.getsize(local) == want["bytes"]:
        print(f"{name}: 本機已有", flush=True)
    elif os.path.isfile(whole):
        if os.path.abspath(whole) != os.path.abspath(local):
            shutil.copyfile(whole, local)
        print(f"{name}: 從 {DRIVE_DIR} 複製（{os.path.getsize(local):,} bytes）", flush=True)
    elif name in pm:
        info, full = pm[name], hashlib.sha256()
        with open(local, "wb") as w:
            for pn, size, h in info["parts"]:
                with open(os.path.join(DRIVE_DIR, "parts", pn), "rb") as g:
                    b = g.read()
                assert len(b) == size and hashlib.sha256(b).hexdigest() == h, f"{pn}: 分塊大小或 SHA-256 不符"
                full.update(b); w.write(b)
        assert full.hexdigest() == info["sha256"] == want["sha256"], f"{name}: 接回後 SHA-256 不符"
        print(f"{name}: 由 {len(info['parts'])} 塊接回，SHA-256 相符", flush=True)
    else:
        raise AssertionError(f"{DRIVE_DIR} 裡沒有 {name}，也沒有它的分塊（parts/ ＋ {PARTS_MANIFEST}）")
    assert os.path.getsize(local) == want["bytes"], f"{name}: {os.path.getsize(local):,} bytes，應為 {want['bytes']:,}（下載／上傳不完整？）"
TIMES["fetch zips"] = time.time() - t0
print(f"{TIMES['fetch zips']:.0f} 秒")

In [ ]:
# 3. repro/stage_inputs.py：在 OFFICIAL_DIR 找 10 個官方測試 CSV（核 SHA-256）→ 複製到 repo 原路徑並重建 6 個衍生檔
#    → 核每個包的 SHA-256 → 解壓到 repo 原路徑（每個檔核 bytes 與 SHA-256）
#    → 核 repo 內的 54 個輸入檔 → 把 C10_features_layer1_ingit.zip 的 67 個檔在這個 clone 裡 git add -f 並 commit
#    （競賽二的凍結工具要求輸入檔已 commit；這個 commit 只在這台 Colab，不推回）→ 確認 git 工作樹乾淨
#    只用標準函式庫，所以用 Colab 內建的 python3 跑。
import sys
t0 = time.time()
p = subprocess.run([sys.executable, "-u", os.path.join(WORK, "repro", "stage_inputs.py"), "--root", WORK, "--zips-dir", ZIP_LOCAL,
                    "--official-dir", OFFICIAL_DIR],
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(p.stdout)
assert p.returncode == 0, "stage_inputs.py 沒有完成，停止"
STAGED_HEAD = subprocess.run(["git", "-C", WORK, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert not os.path.exists(os.path.join(WORK, "data", "local")), "樹裡不應該有 data/local"
shutil.rmtree(ZIP_LOCAL, ignore_errors=True)     # 釋出本機磁碟
TIMES["stage inputs"] = time.time() - t0
print(f"公開 repo 的 commit {HEAD}；本機補回輸入檔後的 commit {STAGED_HEAD}（只在這台機器）；{TIMES['stage inputs']:.0f} 秒")

In [ ]:
# 4. 建 Python 3.12.13 環境（uv），裝 repro/requirements_cpu.txt 的版本
#    requirements_cpu.txt 是 Windows 上 .venv 的實際套件清單。這裡排除：
#      colorama（Windows 上 pytest 的相依；Linux 不需要）
#    清單裡沒有 pywin32 之類只能在 Windows 安裝的套件。
#    torch 是 CPU 版（+cpu），要從 PyTorch 的 CPU 索引抓；第 1 層其實不 import torch。
#    整份裝不起來時（例如某個版本沒有 Linux 輪子），退回只裝第 1 層實際用到的 numpy、scipy、psutil（版本照清單），並印出說明。
import subprocess, re
t0 = time.time()
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-4000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"exit {r.returncode}: {cmd}")
    return r.returncode

sh("pip install -q uv")
sh("uv python install 3.12.13")
sh(f"uv venv {VENV} --python 3.12.13")
PY = f"{VENV}/bin/python"
WINDOWS_ONLY = {"colorama"}
req_src = os.path.join(WORK, "repro", "requirements_cpu.txt")
lines = [l.strip() for l in open(req_src, encoding="utf-8") if l.strip() and not l.startswith("#")]
keep = [l for l in lines if re.split(r"[=<>!~]", l)[0].lower() not in WINDOWS_ONLY]
os.makedirs(OUT, exist_ok=True)
req = os.path.join(OUT, "requirements_linux.txt")
open(req, "w").write("\n".join(keep) + "\n")
INSTALL = "full"
rc = sh(f"uv pip install --python {PY} -r {req} --extra-index-url https://download.pytorch.org/whl/cpu "
        f"--index-strategy unsafe-best-match", check=False)
if rc != 0:
    INSTALL = "minimal (numpy, scipy, psutil)"
    mini = [l for l in keep if re.split(r"[=<>!~]", l)[0].lower() in ("numpy", "scipy", "psutil")]
    print("整份 requirements 裝不起來，改裝第 1 層實際用到的：", mini)
    sh(f"uv pip install --python {PY} " + " ".join(mini))
sh(f"{PY} -c \"import sys, numpy, scipy; print(sys.version.split()[0], numpy.__version__, scipy.__version__)\"")
TIMES["environment"] = time.time() - t0
print("安裝方式：", INSTALL, f"；{TIMES['environment']:.0f} 秒")

In [ ]:
# 5. 跑兩支入口（競賽二先、競賽一後），輸出在 repo 樹之外
#    每支先不加旗標跑一次（逐位元組模式）。若在接縫停下、沒有產生最終 CSV，再加 --tolerant 跑一次（容許模式）。
#    PASS 一律只看最終 CSV 的 SHA-256。
import csv, json, hashlib
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

RESULT = {}
EXPECT = {
    "comp2": ("repro/comp2_from_features.py", "rank/submission_rank.csv", "results/test69/rank/submission_rank.csv",
              "c7595f2eb43c185973c55c7b4b03875de41c5ce92327f7042fc3769dbbae36c8"),
    "comp1": ("repro/comp1_from_features.py", "submission_player.csv", "results/prereg67/apply/submission_player.csv",
              "36b222d866267abffda03d126d94db9ed5bea3406b2f8b47323622fd5329a026"),
}

def child_env():
    env = dict(os.environ)
    for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",   # 入口要求不設（comp2 入口會檢查）
              "PYTHONPATH", "PYTHONHOME"):                                    # 不讓 Colab 內建 Python 的套件混進 3.12.13 環境
        env.pop(v, None)
    env["PYTHONDONTWRITEBYTECODE"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    return env

def rows(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.reader(f))

def diff_rows(got, ref):
    a, b = rows(got), rows(ref)
    n = sum(1 for x, y in zip(a, b) if x != y) + abs(len(a) - len(b))
    return n, len(a), len(b), [(i, x, y) for i, (x, y) in enumerate(zip(a, b)) if x != y][:10]

def run_once(comp, tolerant):
    script, rel_out, rel_ref, want = EXPECT[comp]
    tag = comp + ("_tolerant" if tolerant else "")
    out = os.path.join(OUT, tag)
    shutil.rmtree(out, ignore_errors=True)
    cmd = [PY, "-u", os.path.join(WORK, script), "--root", WORK, "--out", out] + (["--tolerant"] if tolerant else [])
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=WORK, env=child_env(),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    log = []
    for line in p.stdout:
        log.append(line)
        print(line, end="", flush=True)
    rc = p.wait()
    sec = time.time() - t0
    open(os.path.join(OUT, f"{tag}_log.txt"), "w", encoding="utf-8").write("".join(log))
    got = os.path.join(out, *rel_out.split("/"))
    ref = os.path.join(WORK, *rel_ref.split("/"))
    res = {"mode": "tolerant" if tolerant else "bytewise", "exit_code": rc, "seconds": round(sec, 1),
           "expected_sha256": want, "produced": os.path.isfile(got)}
    stop = [l.strip() for l in log if l.startswith(("DRIFT", "REFUSED", "FAIL", "WATCHDOG")) or "SystemExit" in l]
    if rc != 0 and not res["produced"]:
        res["stopped_at"] = (stop or [l.strip() for l in log[-3:]])[:3]
    if res["produced"]:
        res["sha256"] = sha256(got)
        res["pass"] = res["sha256"] == want and rc == 0
        if res["sha256"] != want:
            n, la, lb, ex = diff_rows(got, ref)
            res.update(rows_different=n, rows_produced=la, rows_uploaded=lb, first_differences=ex)
    else:
        res["pass"] = False
    if tolerant and os.path.isfile(os.path.join(out, "tolerant_report.json")):
        tr = json.load(open(os.path.join(out, "tolerant_report.json"), encoding="utf-8"))
        res["tolerant_report"] = tr
        shutil.copy(os.path.join(out, "tolerant_report.json"), os.path.join(OUT, f"{comp}_tolerant_report.json"))
    TIMES[tag] = sec
    print("=" * 100)
    print(f"{comp}（{'容許模式 --tolerant' if tolerant else '逐位元組模式'}）: {'PASS' if res['pass'] else 'FAIL'}   exit code {rc}   {sec:.1f} 秒")
    print(f"  sha256   {res.get('sha256', '(沒有產生最終 CSV：在某個檢查或接縫就停了，看上面的訊息)')}")
    print(f"  expected {want}")
    for l in res.get("stopped_at", []):
        print(f"  停在：{l[:300]}")
    if "rows_different" in res:
        print(f"  和已上傳檔不同的列數：{res['rows_different']}（產生 {res['rows_produced']} 列含表頭，已上傳 {res['rows_uploaded']} 列）")
        for i, x, y in res["first_differences"]:
            print(f"    第 {i} 列  產生 {x}  已上傳 {y}")
    if "tolerant_report" in res:
        tr = res["tolerant_report"]
        print(f"  容許模式：檢查 {tr['seams_checked']} 個接縫、放寬 {tr['seams_relaxed']} 個；數值接縫最大絕對差 {tr['numeric_max_abs_diff']:.3g}"
              f"（都在容許值內：{tr['numeric_all_within_tolerance']}）；離散接縫不同列數 {tr['discrete_rows_different']}")
        for e in tr["seams"]:
            what = "相同" if e["strict_equal"] else ("放寬" if e["within_tolerance"] else "超過容許值／有不同列")
            nums = "  ".join(f"{k} {e[k]:.3g}" if isinstance(e[k], float) else f"{k} {e[k]}"
                             for k in ("max_abs_diff", "max_rel_diff", "rows_different") if e.get(k) is not None)
            print(f"    [{what}] {e['seam']}{'（' + e['context'] + '）' if e.get('context') else ''}  {nums}")
    print("=" * 100, flush=True)
    return res

def run(comp):
    strict = run_once(comp, False)
    RESULT[comp] = {"bytewise": strict, "pass_bytewise": strict["pass"], "tolerant": None, "pass_tolerant": None}
    if not strict["produced"]:
        print(f"{comp}: 逐位元組模式沒有產生最終 CSV，改用 --tolerant 再跑一次", flush=True)
        tol = run_once(comp, True)
        RESULT[comp].update(tolerant=tol, pass_tolerant=tol["pass"])
    RESULT[comp]["pass"] = bool(strict["pass"] or RESULT[comp]["pass_tolerant"])

run("comp2")

In [ ]:
run("comp1")

In [ ]:
# 6. 總結：版本、CPU、耗時
import platform
ver = subprocess.run([PY, "-c", "import sys, numpy, scipy, json; print(json.dumps({'python': sys.version.split()[0], 'numpy': numpy.__version__, 'scipy': scipy.__version__}))"],
                     capture_output=True, text=True, env=child_env()).stdout.strip()
cpu = "?"
try:
    cpu = [l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
except Exception:
    pass
mem = "?"
try:
    mem = [l.split()[1] for l in open("/proc/meminfo") if l.startswith("MemTotal")][0]
    mem = f"{int(mem) / 1048576:.1f} GB"
except Exception:
    pass
blas = subprocess.run([PY, "-c", "import numpy; numpy.show_config()"], capture_output=True, text=True, env=child_env()).stdout
summary = {"versions": json.loads(ver) if ver else None, "install": INSTALL, "platform": platform.platform(),
           "cpu": cpu, "logical_cpus": os.cpu_count(), "ram": mem,
           "seconds": {k: round(v, 1) for k, v in TIMES.items()}, "seconds_total": round(time.time() - T_START, 1),
           "repo": REPO_URL, "commit": HEAD, "commit_after_stage_inputs": STAGED_HEAD, "result": RESULT}
print(json.dumps(summary, ensure_ascii=False, indent=1))
for comp in ("comp2", "comp1"):
    r = RESULT.get(comp, {})
    for mode in ("bytewise", "tolerant"):
        m = r.get(mode)
        if m is None:
            print(f"{comp} {mode}: 沒有跑" + ("（逐位元組模式已產生最終 CSV）" if mode == "tolerant" and r else ""))
            continue
        extra = ""
        if "rows_different" in m:
            extra = f"  和已上傳檔不同 {m['rows_different']} 列"
        if not m["produced"]:
            extra = "  沒有產生最終 CSV：" + "；".join(x[:120] for x in m.get("stopped_at", []))
        if "tolerant_report" in m:
            tr = m["tolerant_report"]
            extra += f"  （放寬 {tr['seams_relaxed']} 個接縫，數值最大差 {tr['numeric_max_abs_diff']:.3g}，離散不同列 {tr['discrete_rows_different']}）"
        print(f"{comp} {mode}: {'PASS' if m['pass'] else 'FAIL'}  {m.get('sha256', '')}  {m['seconds']} 秒{extra}")
    print(f"{comp}: 最終 CSV 和已上傳檔 SHA-256 相同＝{bool(r.get('pass'))}（逐位元組模式 {r.get('pass_bytewise')}、容許模式 {r.get('pass_tolerant')}）")
open(os.path.join(OUT, "colab_public_summary.json"), "w", encoding="utf-8").write(json.dumps(summary, ensure_ascii=False, indent=1))
open(os.path.join(OUT, "numpy_config.txt"), "w", encoding="utf-8").write(blas)
# 把紀錄和產生的 CSV 存回 DRIVE_DIR（只有幾百 KB）；資料夾唯讀（例如別人分享的資料夾）就略過
try:
    if not SAVE_TO_DRIVE:
        raise OSError("SAVE_TO_DRIVE = False")
    dst = os.path.join(DRIVE_DIR, "colab_run_" + time.strftime("%Y%m%d_%H%M%S"))
    os.makedirs(dst, exist_ok=True)
    for n in ("colab_public_summary.json", "numpy_config.txt", "comp1_log.txt", "comp2_log.txt", "comp1_tolerant_log.txt",
              "comp2_tolerant_log.txt", "comp1_tolerant_report.json", "comp2_tolerant_report.json", "requirements_linux.txt"):
        if os.path.isfile(os.path.join(OUT, n)):
            shutil.copy(os.path.join(OUT, n), dst)
    for comp, (_, rel_out, _, _) in EXPECT.items():
        for tag in (comp, comp + "_tolerant"):
            p = os.path.join(OUT, tag, *rel_out.split("/"))
            if os.path.isfile(p):
                shutil.copy(p, os.path.join(dst, f"{tag}_" + os.path.basename(p)))
    print("紀錄存到", dst)
except OSError as e:
    print(f"沒有存回 Drive（{e}）；紀錄在 {OUT}")